**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 26b: Rejection Sampling + SFT 실습

## 🎯 Rejection Sampling이란?

**Rejection Sampling (Best-of-N Sampling)**은 강화학습 없이도 모델 성능을 향상시키는 강력한 기법입니다.

### 핵심 아이디어
```
1. 같은 프롬프트에 대해 모델이 N개의 응답을 생성
2. 보상 모델(또는 LLM Judge)로 각 응답에 점수 부여
3. 가장 높은 점수의 응답만 선택
4. 선택된 고품질 응답으로 SFT 학습!
```

### 왜 Rejection Sampling인가?

| 방법 | 장점 | 단점 |
|------|------|------|
| 일반 SFT | 간단함 | 데이터 품질에 의존 |
| **Rejection Sampling + SFT** | **자체 모델에서 고품질 데이터 추출** | 생성 비용 (N배) |
| DPO/PPO | 직접 선호도 학습 | 복잡한 설정 필요 |

### DeepSeek R1에서의 활용
DeepSeek R1 파이프라인에서도 **RS + SFT**가 핵심 단계입니다:
```
Cold Start → GRPO → Rejection Sampling + SFT → GRPO → Final RL
```

### 학습 목표
- ✅ Rejection Sampling의 개념과 원리 이해
- ✅ 모델에서 N개 응답 생성 (temperature sampling)
- ✅ LLM-as-a-Judge로 응답 품질 평가
- ✅ Best-of-N 선택으로 고품질 SFT 데이터 구축
- ✅ 선택된 데이터로 SFT 학습 및 성능 비교

### 실습 환경
- 모델: Qwen2.5-1.5B-Instruct (4bit)
- 평가: GPT-4o-mini (LLM Judge)
- GPU: RTX 4060 (8GB) 호환
- 예상 VRAM: ~3-4GB

## 1️⃣ 환경 설정 및 모델 로드

In [ ]:
# 필수 라이브러리 임포트
import torch
import gc
import os
import json
import time
import random
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from openai import OpenAI

# OpenAI 클라이언트 (LLM Judge용)
client = OpenAI()

print("✅ 라이브러리 임포트 완료")


In [ ]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")


In [ ]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./output/rejection_sampling_sft"

# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 토크나이저 로드
print("⏳ 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델 로드
print("⏳ 모델 로딩 중... (4bit 양자화)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

print(f"✅ 모델 로드 완료: {MODEL_NAME}")
print_gpu_memory("모델 로드 후")


## 2️⃣ 프롬프트 준비 및 N개 응답 생성

Rejection Sampling의 첫 번째 단계: **같은 프롬프트에 대해 여러 응답을 생성**합니다.

### 핵심 파라미터
- **N (생성 개수)**: 보통 4~16개. N이 클수록 좋은 응답 나올 확률 ↑, 하지만 비용도 ↑
- **temperature**: 0.7~1.0 권장. 너무 낮으면 다양성 ↓, 너무 높으면 품질 ↓
- **top_p**: 0.9~0.95로 약간의 다양성 확보

In [ ]:
# 학습용 프롬프트 준비
prompts = [
    "머신러닝과 딥러닝의 차이점을 설명하세요.",
    "좋은 프롬프트를 작성하는 방법을 알려주세요.",
    "Python에서 리스트와 튜플의 차이점은 무엇인가요?",
    "트랜스포머 모델의 어텐션 메커니즘을 설명하세요.",
    "오버피팅을 방지하는 방법들을 나열하세요.",
    "REST API와 GraphQL의 차이를 비교하세요.",
    "Git에서 rebase와 merge의 차이를 설명하세요.",
    "데이터 정규화가 왜 중요한지 설명하세요.",
    "LoRA 파인튜닝의 장점을 설명하세요.",
    "RAG(Retrieval-Augmented Generation)의 원리를 설명하세요.",
    "경사 하강법(Gradient Descent)의 원리를 설명하세요.",
    "LLM의 환각(Hallucination) 문제와 해결 방법을 설명하세요.",
    "배치 정규화(Batch Normalization)의 역할을 설명하세요.",
    "Docker와 가상머신의 차이를 설명하세요.",
    "학습률(Learning Rate)이 모델 학습에 미치는 영향을 설명하세요.",
]

print(f"✅ 프롬프트 준비 완료: {len(prompts)}개")
for i, p in enumerate(prompts[:5]):
    print(f"  {i+1}. {p}")
print(f"  ... (총 {len(prompts)}개)")


In [ ]:
# N개 응답 생성 함수
def generate_n_responses(model, tokenizer, prompt, n=8, max_new_tokens=256,
                         temperature=0.8, top_p=0.9):
    """같은 프롬프트에 대해 N개의 다양한 응답을 생성"""
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다. 질문에 정확하고 도움이 되는 답변을 하세요."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    responses = []
    for i in range(n):
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        responses.append(response.strip())
    
    return responses

# 테스트: 첫 번째 프롬프트로 응답 다양성 확인
print("📊 응답 다양성 테스트 (첫 번째 프롬프트)")
print("="*60)
print(f"프롬프트: {prompts[0]}")
print("="*60)

test_responses = generate_n_responses(model, tokenizer, prompts[0], n=4)
for i, resp in enumerate(test_responses):
    print(f"\n🔹 응답 {i+1}:")
    print(f"  {resp[:200]}...")
    print(f"  (길이: {len(resp)}자)")


In [ ]:
# 전체 프롬프트에 대해 N개 응답 생성
N = 8  # 프롬프트당 생성할 응답 수

print(f"🚀 전체 {len(prompts)}개 프롬프트 x {N}개 응답 = {len(prompts) * N}개 생성 시작!")
print("="*60)

all_candidates = []

for idx, prompt in enumerate(prompts):
    print(f"⏳ [{idx+1}/{len(prompts)}] {prompt[:40]}...")
    responses = generate_n_responses(model, tokenizer, prompt, n=N)
    all_candidates.append({
        "prompt": prompt,
        "responses": responses,
    })
    print(f"   ✅ {len(responses)}개 응답 생성 완료")

print("="*60)
print(f"✅ 전체 응답 생성 완료: {sum(len(c['responses']) for c in all_candidates)}개")


## 3️⃣ LLM-as-a-Judge로 응답 평가 (GPT-4o-mini)

Rejection Sampling의 두 번째 단계: **각 응답에 점수를 매깁니다.**

### 평가 기준
- **정확성** (1-5): 내용이 사실적이고 올바른가?
- **완성도** (1-5): 충분히 자세하고 구체적인가?
- **명확성** (1-5): 이해하기 쉽고 잘 구조화되어 있는가?
- **종합 점수**: 3개 점수의 평균

### 왜 GPT-4o-mini를 Judge로 사용하나?
- 비용: 1건당 ~$0.001 (매우 저렴)
- 속도: 빠른 응답
- 품질: 1.5B 모델 응답 평가에는 충분한 수준

In [ ]:
# LLM Judge 함수 (GPT-4o-mini)
def score_response(prompt, response, model_name="gpt-4o-mini"):
    """GPT-4o-mini로 응답 품질을 1-5점으로 평가"""
    judge_prompt = f"""당신은 AI 응답 품질을 평가하는 전문 평가자입니다.
아래의 질문과 응답을 읽고, 3가지 기준으로 1-5점을 매겨주세요.

## 질문
{prompt}

## 응답
{response}

## 평가 기준
1. 정확성 (1-5): 내용이 사실적이고 올바른가?
2. 완성도 (1-5): 충분히 자세하고 구체적인가?
3. 명확성 (1-5): 이해하기 쉽고 잘 구조화되어 있는가?

반드시 아래 JSON 형식으로만 답변하세요:
{{"accuracy": 점수, "completeness": 점수, "clarity": 점수, "reason": "한 줄 평가 이유"}}"""

    try:
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.1,
            response_format={"type": "json_object"},
        )
        result = json.loads(resp.choices[0].message.content)
        avg_score = (result["accuracy"] + result["completeness"] + result["clarity"]) / 3
        result["avg_score"] = round(avg_score, 2)
        return result
    except Exception as e:
        print(f"  ⚠️ 평가 실패: {e}")
        return {"accuracy": 3, "completeness": 3, "clarity": 3, "avg_score": 3.0, "reason": "평가 실패"}

# 테스트
print("📊 LLM Judge 테스트")
print("="*60)
test_score = score_response(prompts[0], test_responses[0])
print(f"프롬프트: {prompts[0]}")
print(f"응답: {test_responses[0][:100]}...")
print(f"평가: {json.dumps(test_score, ensure_ascii=False, indent=2)}")


In [ ]:
# 전체 응답 평가
print("🚀 전체 응답 LLM Judge 평가 시작!")
print("="*60)

scored_candidates = []

for idx, candidate in enumerate(all_candidates):
    prompt = candidate["prompt"]
    print(f"\n⏳ [{idx+1}/{len(all_candidates)}] {prompt[:40]}...")
    
    scored_responses = []
    for j, response in enumerate(candidate["responses"]):
        score = score_response(prompt, response)
        scored_responses.append({
            "response": response,
            "score": score,
        })
        print(f"   응답 {j+1}: {score['avg_score']:.1f}점 ({score.get('reason', '')[:30]})")
    
    # 점수 순으로 정렬
    scored_responses.sort(key=lambda x: x["score"]["avg_score"], reverse=True)
    scored_candidates.append({
        "prompt": prompt,
        "scored_responses": scored_responses,
        "best_score": scored_responses[0]["score"]["avg_score"],
        "worst_score": scored_responses[-1]["score"]["avg_score"],
    })
    print(f"   📊 최고: {scored_responses[0]['score']['avg_score']:.1f}점 / 최저: {scored_responses[-1]['score']['avg_score']:.1f}점")

print("\n" + "="*60)
print("✅ 전체 평가 완료!")


## 4️⃣ Best-of-N 선택 — 고품질 SFT 데이터 구축

핵심 단계: **각 프롬프트에서 가장 높은 점수의 응답만 선택**합니다.

### 선택 전략
- **Best-of-N**: 최고 점수 1개만 선택 (가장 기본)
- **Top-K**: 상위 K개 선택 (데이터 양 확보)
- **Threshold**: 특정 점수 이상만 선택 (품질 보장)

이번 실습에서는 **Best-of-N (최고 1개)** + **Threshold (4.0점 이상 추가 선택)**을 결합합니다.

In [ ]:
# Best-of-N 선택 + Threshold 필터링
SCORE_THRESHOLD = 4.0  # 이 점수 이상의 응답도 추가 선택

selected_data = []
rejected_data = []  # 비교용 (최저 점수 응답)

print("📊 Best-of-N 선택 결과")
print("="*60)

for candidate in scored_candidates:
    prompt = candidate["prompt"]
    responses = candidate["scored_responses"]
    
    # Best (최고 점수)
    best = responses[0]
    selected_data.append({
        "prompt": prompt,
        "response": best["response"],
        "score": best["score"]["avg_score"],
        "selection": "best-of-N",
    })
    
    # Threshold (추가 선택: best 제외, 4.0 이상)
    for resp in responses[1:]:
        if resp["score"]["avg_score"] >= SCORE_THRESHOLD:
            selected_data.append({
                "prompt": prompt,
                "response": resp["response"],
                "score": resp["score"]["avg_score"],
                "selection": "threshold",
            })
    
    # Worst (비교용)
    worst = responses[-1]
    rejected_data.append({
        "prompt": prompt,
        "response": worst["response"],
        "score": worst["score"]["avg_score"],
    })
    
    print(f"📌 {prompt[:40]}...")
    print(f"   Best: {best['score']['avg_score']:.1f}점 / Worst: {worst['score']['avg_score']:.1f}점 / 차이: {best['score']['avg_score'] - worst['score']['avg_score']:.1f}")

print("\n" + "="*60)
print(f"✅ 선택 완료!")
print(f"   📄 선택된 고품질 데이터: {len(selected_data)}건")
print(f"   📄 비교용 저품질 데이터: {len(rejected_data)}건")
print(f"   📊 평균 점수 (선택): {sum(d['score'] for d in selected_data)/len(selected_data):.2f}")
print(f"   📊 평균 점수 (탈락): {sum(d['score'] for d in rejected_data)/len(rejected_data):.2f}")


In [ ]:
# 점수 분포 분석
print("📊 점수 분포 분석")
print("="*60)

all_scores = []
for candidate in scored_candidates:
    for resp in candidate["scored_responses"]:
        all_scores.append(resp["score"]["avg_score"])

selected_scores = [d["score"] for d in selected_data]
rejected_scores = [d["score"] for d in rejected_data]

# 히스토그램 (텍스트)
for label, scores in [("전체", all_scores), ("선택", selected_scores), ("탈락(worst)", rejected_scores)]:
    print(f"\n🔹 {label} 응답 ({len(scores)}건):")
    print(f"   평균: {sum(scores)/len(scores):.2f} / 최고: {max(scores):.1f} / 최저: {min(scores):.1f}")
    
    for threshold in [5.0, 4.5, 4.0, 3.5, 3.0, 2.5, 2.0, 1.5, 1.0]:
        count = sum(1 for s in scores if s >= threshold and s < threshold + 0.5)
        bar = "█" * count
        if count > 0:
            print(f"   {threshold:.1f}~{threshold+0.4:.1f}: {bar} ({count})")

# matplotlib 시각화 (있으면)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    ax.hist(all_scores, bins=20, alpha=0.5, label=f"전체 ({len(all_scores)}개)", color="gray")
    ax.hist(selected_scores, bins=20, alpha=0.7, label=f"선택 ({len(selected_scores)}개)", color="green")
    ax.axvline(x=SCORE_THRESHOLD, color="red", linestyle="--", label=f"Threshold ({SCORE_THRESHOLD})")
    ax.set_xlabel("점수")
    ax.set_ylabel("개수")
    ax.set_title("Rejection Sampling: 점수 분포")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print("✅ 점수 분포 차트 생성 완료")
except ImportError:
    print("ℹ️ matplotlib 미설치 — 텍스트 분포만 표시")


## 5️⃣ 선택된 데이터로 SFT 학습

이제 Rejection Sampling으로 선별한 **고품질 데이터**로 모델을 학습합니다.

### 기대 효과
- 모델 자체에서 생성한 응답 → **분포가 자연스러움**
- 최고 품질만 선별 → **학습 효율 극대화**
- DPO처럼 복잡한 설정 없이 **일반 SFT로 가능**

In [ ]:
# 학습 전 응답 저장 (Before)
print("="*60)
print("📋 학습 전 모델 응답 (Before Training)")
print("="*60)

EVAL_PROMPTS = [
    "오버피팅과 언더피팅의 차이를 설명하세요.",
    "Transformer 모델에서 Self-Attention이 중요한 이유를 설명하세요.",
    "좋은 코드 리뷰를 하는 방법을 알려주세요.",
    "LLM 파인튜닝에서 LoRA를 사용하는 이유를 설명하세요.",
]

def generate_eval_response(model, tokenizer, prompt):
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=256, temperature=0.1,
            do_sample=True, pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

before_responses = []
for i, prompt in enumerate(EVAL_PROMPTS):
    resp = generate_eval_response(model, tokenizer, prompt)
    before_responses.append(resp)
    print(f"\n🔹 질문 {i+1}: {prompt}")
    print(f"   답변: {resp[:200]}...")


In [ ]:
# 학습 데이터 포맷팅 (Chat Template)
def format_to_chat(sample):
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다. 정확하고 친절하게 답변해주세요."},
        {"role": "user", "content": sample["prompt"]},
        {"role": "assistant", "content": sample["response"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return text

formatted_texts = [format_to_chat(s) for s in selected_data]
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"✅ 학습 데이터 포맷팅 완료: {len(dataset)}개")
print(f"\n--- 포맷팅 예시 ---")
print(dataset[0]["text"][:300])


In [ ]:
# LoRA 설정 및 적용
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print_gpu_memory("LoRA 적용 후")


In [ ]:
# SFTTrainer 설정 및 학습
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=False,
    logging_steps=1,
    save_strategy="no",
    max_length=1024,
    gradient_checkpointing=True,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("✅ SFTTrainer 초기화 완료")
print_gpu_memory("학습 시작 전")


In [ ]:
# 학습 실행
print("🚀 Rejection Sampling SFT 학습 시작!")
print("="*60)

start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

print("="*60)
print("✅ 학습 완료!")
print(f"📌 Total steps: {train_result.global_step}")
print(f"📌 Training loss: {train_result.training_loss:.4f}")
print(f"📌 학습 시간: {training_time:.1f}초")
print_gpu_memory("학습 완료 후")


## 6️⃣ 학습 전후 비교

Rejection Sampling으로 선별한 고품질 데이터로 학습한 결과를 확인합니다.

In [17]:
# 학습 후 응답 생성 및 비교
print("="*60)
print("📊 학습 전후 비교 (Before vs After)")
print("="*60)

model.eval()
model.half()  # float16 통일 (추론 시 dtype 불일치 방지)

after_responses = []

for i, prompt in enumerate(EVAL_PROMPTS):
    resp = generate_eval_response(model, tokenizer, prompt)
    after_responses.append(resp)
    
    print(f"\n{'='*60}")
    print(f"🔹 질문 {i+1}: {prompt}")
    print(f"\n📌 Before:")
    print(f"   {before_responses[i][:250]}")
    print(f"\n📌 After (RS+SFT):")
    print(f"   {resp[:250]}")


📊 학습 전후 비교 (Before vs After)

🔹 질문 1: 오버피팅과 언더피팅의 차이를 설명하세요.

📌 Before:
   "오버피팅"과 "언더피팅"은 딥러닝에서 사용되는 두 가지 중요한 개념 중 하나입니다.

1. 오버피팅: 
   - 이는 모델이 학습할 때 데이터가 너무 많거나 다양하다면 발생하는 현상입니다.
   - 이는 모델이 학습할 때 모든 특징을 학습하게 되어, 새로운 데이터에 대해서도 잘 예측하거나 분류하지 못하게 될 수 있습니다.
   - 이를 해결하기 위해 일반적으로 더 적은 데이터셋을 사용하거나 더 복잡한 모델을 사용해야 합니다.

2. 언더피팅:

📌 After (RS+SFT):
   "오버피팅"과 "언더피팅"은 딥러닝에서 사용되는 두 가지 개념 중 하나로, 데이터셋에 대한 모델의 학습을 조절하는 방법에 따라 관련이 있습니다.

1. 오버피팅: 
   - 이는 모델이 학습할 때 더 많은 데이터가 필요하다고 느껴질 때 발생합니다.
   - 이는 일반적으로 모델이 학습하고 있는 데이터와 매우 비슷하거나 동일한 특징만을 학습하게 되어, 새로운 데이터에 대해 예측력이 떨어집니다.
   - 이를 해결하기 위해, 모델의 복잡성을 줄이고

🔹 질문 2: Transformer 모델에서 Self-Attention이 중요한 이유를 설명하세요.

📌 Before:
   Self-Attention은 Transformer 모델에서 매우 중요하며, 이는 각 단위(또는 토큰) 간의 상호작용을 통해 정보를 효과적으로 처리하는 데 도움이 됩니다.

Self-Attention은 각 토큰에 대해 다른 토큰과의 상호작용을 통해 각각의 토큰에 대한 정보를 복잡하게 조정하고, 이를 통해 각 토큰 간의 상호작용을 가능하게 합니다. 

예를 들어, 한 번의 텍스트 입력에서 각 단어가 다른 단어와의 상호작용을 통해 의미를 얻고, 이러

📌 After (RS+SFT):
   Self-Attention은 Transformer 모델의 핵심 기능 중 하나로, 각 단위(또는 토큰) 

In [18]:
# LLM Judge로 Before/After 정량 비교
print("\n" + "="*60)
print("📊 LLM Judge 정량 비교 (Before vs After)")
print("="*60)

before_scores = []
after_scores = []

for i, prompt in enumerate(EVAL_PROMPTS):
    b_score = score_response(prompt, before_responses[i])
    a_score = score_response(prompt, after_responses[i])
    before_scores.append(b_score["avg_score"])
    after_scores.append(a_score["avg_score"])
    
    diff = a_score["avg_score"] - b_score["avg_score"]
    emoji = "📈" if diff > 0 else ("📉" if diff < 0 else "➡️")
    print(f"\n{emoji} 질문 {i+1}: {prompt[:40]}...")
    print(f"   Before: {b_score['avg_score']:.1f}점 / After: {a_score['avg_score']:.1f}점 ({diff:+.1f})")

print(f"\n{'='*60}")
print(f"📊 평균 점수:")
print(f"   Before: {sum(before_scores)/len(before_scores):.2f}")
print(f"   After:  {sum(after_scores)/len(after_scores):.2f}")
print(f"   변화:   {sum(after_scores)/len(after_scores) - sum(before_scores)/len(before_scores):+.2f}")



📊 LLM Judge 정량 비교 (Before vs After)

📉 질문 1: 오버피팅과 언더피팅의 차이를 설명하세요....
   Before: 3.7점 / After: 3.3점 (-0.3)

📉 질문 2: Transformer 모델에서 Self-Attention이 중요한 이유를...
   Before: 4.3점 / After: 3.7점 (-0.7)

📉 질문 3: 좋은 코드 리뷰를 하는 방법을 알려주세요....
   Before: 4.0점 / After: 3.7점 (-0.3)

➡️ 질문 4: LLM 파인튜닝에서 LoRA를 사용하는 이유를 설명하세요....
   Before: 3.7점 / After: 3.7점 (+0.0)

📊 평균 점수:
   Before: 3.92
   After:  3.58
   변화:   -0.33


## 7️⃣ 모델 저장 및 데이터 저장

In [19]:
# LoRA 어댑터 저장
save_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ 어댑터 저장 완료: {save_path}")

# 저장 크기 확인
total_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fns in os.walk(save_path)
    for f in fns
)
print(f"📌 어댑터 크기: {total_size / 1024 / 1024:.1f} MB")

# Rejection Sampling 데이터 저장
rs_data_path = os.path.join(OUTPUT_DIR, "rejection_sampling_data.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(rs_data_path, "w", encoding="utf-8") as f:
    json.dump({
        "selected": selected_data,
        "rejected": rejected_data,
        "metadata": {
            "model": MODEL_NAME,
            "n_per_prompt": N,
            "threshold": SCORE_THRESHOLD,
            "total_generated": sum(len(c["scored_responses"]) for c in scored_candidates),
            "total_selected": len(selected_data),
        }
    }, f, ensure_ascii=False, indent=2)

print(f"✅ RS 데이터 저장: {rs_data_path}")


✅ 어댑터 저장 완료: ./output/rejection_sampling_sft/lora_adapter
📌 어댑터 크기: 50.4 MB
✅ RS 데이터 저장: ./output/rejection_sampling_sft/rejection_sampling_data.json


In [20]:
# GPU 메모리 정리
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("정리 후")


[정리 후] GPU: 1.2GB / 23.5GB


## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

```
프롬프트 15개
    ↓ N=8 응답 생성 (temperature=0.8)
총 120개 응답
    ↓ LLM Judge (GPT-4o-mini) 평가
점수 매긴 120개 응답
    ↓ Best-of-N + Threshold(4.0) 선택
고품질 SFT 데이터
    ↓ LoRA SFT 학습
성능 향상된 모델!
```

### 핵심 포인트

- 🎯 **Rejection Sampling = 자가 증류(Self-Distillation)**: 모델 자체 출력에서 최고를 골라 다시 학습
- 🎯 **DPO/PPO 없이도 성능 향상 가능**: 일반 SFT만으로 고품질 데이터 효과를 누림
- 🎯 **N이 클수록 좋은 응답 확률 ↑**: 하지만 생성 비용도 N배 (trade-off)
- 🎯 **LLM Judge 비용 저렴**: GPT-4o-mini로 120개 평가 ~$0.12
- 🎯 **DeepSeek R1 파이프라인의 핵심 단계**: Cold Start → GRPO → **RS + SFT** → GRPO → Final

### Rejection Sampling vs 다른 방법

| 방법 | 데이터 | 복잡도 | 효과 |
|------|--------|--------|------|
| 일반 SFT | 외부 데이터 | ⭐ | 기본 |
| **RS + SFT** | **자체 생성 + 선별** | ⭐⭐ | **좋음** |
| DPO | chosen/rejected 쌍 | ⭐⭐⭐ | 매우 좋음 |
| PPO/GRPO | 보상 모델 필요 | ⭐⭐⭐⭐ | 최고 |

### 실무 팁
- 프롬프트 다양성이 중요 (최소 100개 이상 권장)
- N=8~16이 가성비 최적
- Threshold는 데이터 분포 보고 조절 (너무 높으면 데이터 부족)
- RS 데이터 + 기존 SFT 데이터 혼합 학습도 효과적

### 다음 단계
- **Session 28**: DPO 학습 — RS+SFT 위에 선호도 정렬을 추가하면 더 강력!
- RS로 만든 데이터를 DPO의 chosen으로 활용 가능